# Notebook 3 — Sentiment Classification: Classical ML Models
### Logistic Regression · Random Forest · XGBoost

| Item | Detail |
|------|--------|
| **Input** | `Dataset/results/gold/gold_reviews.parquet` (6.99M rows, 45 features) |
| **Task** | Binary sentiment classification (Negative=0 / Positive=1) |
| **Label** | `sentiment_binary` — excludes neutral (3★) reviews |
| **Class balance** | ~26% Negative / ~74% Positive |
| **Output** | Trained models + metrics saved to `Dataset/results/models/` |

## 1 — Install Dependencies

In [1]:
!pip install pyspark pyarrow xgboost scikit-learn matplotlib seaborn joblib -q

## 2 — Imports

In [2]:
import os, warnings, joblib
from pathlib import Path
from functools import reduce

# PySpark
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import FloatType, IntegerType
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    VectorAssembler, StandardScaler,
    Tokenizer, StopWordsRemover, HashingTF, IDF
)
from pyspark.ml.classification import (
    LogisticRegression, RandomForestClassifier
)
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)

# Sklearn + XGBoost (for sampled XGBoost)
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve
)
from sklearn.preprocessing import StandardScaler as SkScaler
from xgboost import XGBClassifier

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})
print('Libraries loaded.')

Libraries loaded.


## 3 — Mount Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 4 — Paths & Spark Session

In [4]:
RESULTS_DIR = '/content/drive/MyDrive/Dataset/results'
GOLD_DIR    = f'{RESULTS_DIR}/gold'
MODELS_DIR  = f'{RESULTS_DIR}/models'
PLOTS_DIR   = f'{RESULTS_DIR}/ml_plots'

for d in [MODELS_DIR, PLOTS_DIR]:
    os.makedirs(d, exist_ok=True)

def save_fig(name):
    path = os.path.join(PLOTS_DIR, f'{name}.png')
    plt.savefig(path, bbox_inches='tight')
    plt.show(); plt.close()
    print(f'  Saved → {path}')

spark = (
    SparkSession.builder
    .appName('YelpML_SentimentClassification')
    .config('spark.driver.memory', '8g')
    .config('spark.driver.maxResultSize', '4g')
    .config('spark.sql.shuffle.partitions', '50')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f'Spark {spark.version} ready.')

Spark 4.0.2 ready.


## 5 — Load Gold Layer

In [5]:
gold = spark.read.parquet(f'{GOLD_DIR}/gold_reviews.parquet')

# Binary classification: drop neutral (3★)
df = gold.filter(F.col('sentiment_binary').isNotNull()) \
         .withColumnRenamed('sentiment_binary', 'label')

total = df.count()
label_dist = df.groupBy('label').count().orderBy('label').toPandas()
label_dist['pct'] = (label_dist['count'] / total * 100).round(2)
label_dist['class'] = label_dist['label'].map({0: 'Negative (1-2★)', 1: 'Positive (4-5★)'})

print(f'Total rows (binary): {total:,}')
print(f'\n{label_dist[["class","count","pct"]].to_string(index=False)}')

Total rows (binary): 6,298,346

          class   count   pct
Negative (1-2★) 1613801 25.62
Positive (4-5★) 4684545 74.38


## 6 — Feature Definition & Null Handling

In [6]:
NUMERIC_FEATURES = [
    # Text features
    'char_count', 'word_count', 'avg_word_length',
    'exclamation_count', 'question_count', 'uppercase_ratio', 'sentence_count',
    # Vote features
    'useful', 'funny', 'cool', 'total_votes',
    # Temporal features
    'review_year', 'review_month', 'review_dow', 'review_hour', 'is_weekend',
    # User features
    'user_review_count', 'user_avg_stars', 'user_useful_votes',
    'user_funny_votes', 'user_cool_votes', 'user_fans',
    'user_is_elite', 'user_elite_years', 'user_tenure_days', 'user_engagement_score',
    # Business features
    'biz_stars', 'biz_review_count', 'biz_is_open',
    'biz_category_count', 'biz_latitude', 'biz_longitude',
    # Interaction features
    'user_star_deviation', 'biz_star_deviation',
]

# Fill nulls for user/business features (left join may produce nulls)
fill_defaults = {c: 0.0 for c in NUMERIC_FEATURES}
df = df.fillna(fill_defaults)

# Class weights to handle imbalance (neg:pos ≈ 1:2.9 → give neg more weight)
neg_count = label_dist[label_dist['label'] == 0]['count'].values[0]
pos_count = label_dist[label_dist['label'] == 1]['count'].values[0]
neg_weight = pos_count / total
pos_weight = neg_count / total

df = df.withColumn('class_weight',
    F.when(F.col('label') == 0, float(neg_weight * 2))
     .otherwise(float(pos_weight * 2))
)

print(f'Numeric features : {len(NUMERIC_FEATURES)}')
print(f'Negative weight  : {neg_weight*2:.4f}')
print(f'Positive weight  : {pos_weight*2:.4f}')

Numeric features : 34
Negative weight  : 1.4875
Positive weight  : 0.5125


## 7 — Train / Test Split (80 / 20)

In [7]:
# Stratified split: split per class then union
neg_df = df.filter(F.col('label') == 0)
pos_df = df.filter(F.col('label') == 1)

neg_train, neg_test = neg_df.randomSplit([0.8, 0.2], seed=42)
pos_train, pos_test = pos_df.randomSplit([0.8, 0.2], seed=42)

train_df = neg_train.union(pos_train).cache()
test_df  = neg_test.union(pos_test).cache()

print(f'Train : {train_df.count():>10,}')
print(f'Test  : {test_df.count():>10,}')

Train :  5,038,395
Test  :  1,259,951


---
## 8 — PySpark ML Feature Pipeline

In [8]:
# --- Tabular feature pipeline ---
assembler = VectorAssembler(
    inputCols=NUMERIC_FEATURES,
    outputCol='raw_features',
    handleInvalid='keep'
)
scaler = StandardScaler(
    inputCol='raw_features',
    outputCol='features',
    withStd=True, withMean=False
)

# --- TF-IDF text pipeline ---
tokenizer    = Tokenizer(inputCol='text', outputCol='tokens')
remover      = StopWordsRemover(inputCol='tokens', outputCol='filtered_tokens')
hashing_tf   = HashingTF(inputCol='filtered_tokens', outputCol='raw_tf', numFeatures=10000)
idf          = IDF(inputCol='raw_tf', outputCol='tfidf_features', minDocFreq=5)

# --- Combined assembler (tabular + TF-IDF) ---
combined_assembler = VectorAssembler(
    inputCols=['features', 'tfidf_features'],
    outputCol='combined_features',
    handleInvalid='keep'
)

print('Feature pipeline stages defined.')

Feature pipeline stages defined.


---
# Model 1 — Logistic Regression

## 9 — LR: Train on Tabular Features

In [9]:
lr = LogisticRegression(
    featuresCol='features',
    labelCol='label',
    weightCol='class_weight',
    maxIter=100,
    regParam=0.01,
    elasticNetParam=0.0,
    family='binomial'
)

lr_tab_pipeline = Pipeline(stages=[assembler, scaler, lr])

print('Training Logistic Regression on tabular features ...')
lr_tab_model = lr_tab_pipeline.fit(train_df)
print('Done.')

Training Logistic Regression on tabular features ...
Done.


## 10 — LR: Train on Tabular + TF-IDF Features

In [10]:
lr_combined = LogisticRegression(
    featuresCol='combined_features',
    labelCol='label',
    weightCol='class_weight',
    maxIter=50,
    regParam=0.1,
    elasticNetParam=0.0,
    family='binomial'
)

lr_comb_pipeline = Pipeline(stages=[
    assembler, scaler,
    tokenizer, remover, hashing_tf, idf,
    combined_assembler, lr_combined
])

print('Training Logistic Regression on tabular + TF-IDF features ...')
lr_comb_model = lr_comb_pipeline.fit(train_df)
print('Done.')

Training Logistic Regression on tabular + TF-IDF features ...
Done.


---
# Model 2 — Random Forest

## 11 — RF: Train on Tabular Features

In [11]:
rf = RandomForestClassifier(
    featuresCol='features',
    labelCol='label',
    weightCol='class_weight',
    numTrees=100,
    maxDepth=10,
    minInstancesPerNode=10,
    seed=42
)

rf_pipeline = Pipeline(stages=[assembler, scaler, rf])

print('Training Random Forest (100 trees, maxDepth=10) ...')
rf_model = rf_pipeline.fit(train_df)
print('Done.')

Training Random Forest (100 trees, maxDepth=10) ...
Done.


---
# Model 3 — XGBoost

## 12 — XGBoost: Train on 500k Sample (sklearn)

In [12]:
# Sample 500k rows (stratified) for XGBoost
SAMPLE_SIZE = 500_000
sample_frac  = SAMPLE_SIZE / total

xgb_sample = (
    df.filter(F.col('label') == 0).sample(fraction=sample_frac, seed=42)
    .union(df.filter(F.col('label') == 1).sample(fraction=sample_frac, seed=42))
    .select(NUMERIC_FEATURES + ['label'])
    .toPandas()
)

xgb_sample = xgb_sample.fillna(0)
X = xgb_sample[NUMERIC_FEATURES].values
y = xgb_sample['label'].values

# Stratified 80/20 split within sample
from sklearn.model_selection import train_test_split
X_train, X_test_xgb, y_train, y_test_xgb = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scale_ratio = (y == 0).sum() / (y == 1).sum()

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_ratio,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
    tree_method='hist'
)

print(f'Training XGBoost on {len(X_train):,} samples ...')
xgb_model.fit(X_train, y_train,
              eval_set=[(X_test_xgb, y_test_xgb)],
              verbose=50)
print('Done.')

Training XGBoost on 400,599 samples ...
[0]	validation_0-logloss:0.59815
[50]	validation_0-logloss:0.00376
[100]	validation_0-logloss:0.00005
[150]	validation_0-logloss:0.00001
[200]	validation_0-logloss:0.00001
[250]	validation_0-logloss:0.00001
[299]	validation_0-logloss:0.00001
Done.


---
# Evaluation

## 13 — Evaluation Helper Functions

In [13]:
binary_eval  = BinaryClassificationEvaluator(labelCol='label', rawPredictionCol='rawPrediction')
acc_eval     = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='accuracy')
f1_eval      = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='f1')
prec_eval    = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='weightedPrecision')
recall_eval  = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='weightedRecall')

def evaluate_spark_model(model, test_data, name):
    preds = model.transform(test_data)
    auc   = binary_eval.evaluate(preds)
    acc   = acc_eval.evaluate(preds)
    f1    = f1_eval.evaluate(preds)
    prec  = prec_eval.evaluate(preds)
    rec   = recall_eval.evaluate(preds)
    print(f'\n{name}')
    print(f'  AUC-ROC   : {auc:.4f}')
    print(f'  Accuracy  : {acc:.4f}')
    print(f'  F1 (wtd)  : {f1:.4f}')
    print(f'  Precision : {prec:.4f}')
    print(f'  Recall    : {rec:.4f}')
    return {'Model': name, 'AUC-ROC': round(auc,4), 'Accuracy': round(acc,4),
            'F1': round(f1,4), 'Precision': round(prec,4), 'Recall': round(rec,4)}, preds

def evaluate_sklearn_model(y_true, y_pred, y_proba, name):
    auc   = roc_auc_score(y_true, y_proba)
    acc   = accuracy_score(y_true, y_pred)
    report = classification_report(y_true, y_pred, output_dict=True)
    f1    = report['weighted avg']['f1-score']
    prec  = report['weighted avg']['precision']
    rec   = report['weighted avg']['recall']
    print(f'\n{name}')
    print(f'  AUC-ROC   : {auc:.4f}')
    print(f'  Accuracy  : {acc:.4f}')
    print(f'  F1 (wtd)  : {f1:.4f}')
    print(f'  Precision : {prec:.4f}')
    print(f'  Recall    : {rec:.4f}')
    return {'Model': name, 'AUC-ROC': round(auc,4), 'Accuracy': round(acc,4),
            'F1': round(f1,4), 'Precision': round(prec,4), 'Recall': round(rec,4)}

print('Evaluators ready.')

Evaluators ready.


## 14 — Evaluate All Models

In [14]:
results = []

print('=== Evaluating models on test set ===')

# LR Tabular
r1, preds_lr_tab = evaluate_spark_model(lr_tab_model,  test_df, 'LR — Tabular Features')
results.append(r1)

# LR Combined
r2, preds_lr_comb = evaluate_spark_model(lr_comb_model, test_df, 'LR — Tabular + TF-IDF')
results.append(r2)

# RF Tabular
r3, preds_rf = evaluate_spark_model(rf_model, test_df, 'Random Forest — Tabular Features')
results.append(r3)

# XGBoost (sklearn on sample)
xgb_preds_proba = xgb_model.predict_proba(X_test_xgb)[:, 1]
xgb_preds       = xgb_model.predict(X_test_xgb)
r4 = evaluate_sklearn_model(y_test_xgb, xgb_preds, xgb_preds_proba, 'XGBoost — Tabular (500k sample)')
results.append(r4)

metrics_df = pd.DataFrame(results)
print('\n=== FINAL METRICS TABLE ===')
display(metrics_df.sort_values('AUC-ROC', ascending=False).reset_index(drop=True))

=== Evaluating models on test set ===

LR — Tabular Features
  AUC-ROC   : 1.0000
  Accuracy  : 1.0000
  F1 (wtd)  : 1.0000
  Precision : 1.0000
  Recall    : 1.0000

LR — Tabular + TF-IDF
  AUC-ROC   : 0.9996
  Accuracy  : 0.9951
  F1 (wtd)  : 0.9952
  Precision : 0.9952
  Recall    : 0.9951

Random Forest — Tabular Features
  AUC-ROC   : 1.0000
  Accuracy  : 1.0000
  F1 (wtd)  : 1.0000
  Precision : 1.0000
  Recall    : 1.0000

XGBoost — Tabular (500k sample)
  AUC-ROC   : 1.0000
  Accuracy  : 1.0000
  F1 (wtd)  : 1.0000
  Precision : 1.0000
  Recall    : 1.0000

=== FINAL METRICS TABLE ===


,Model,AUC-ROC,Accuracy,F1,Precision,Recall
0,LR — Tabular Features,1.0000,1.0000,1.0000,1.0000,1.0000
1,Random Forest — Tabular Features,1.0000,1.0000,1.0000,1.0000,1.0000
2,XGBoost — Tabular (500k sample),1.0000,1.0000,1.0000,1.0000,1.0000
3,LR — Tabular + TF-IDF,0.9996,0.9951,0.9952,0.9952,0.9951


## 15 — Model Comparison Plot

In [15]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Model Comparison — Sentiment Binary Classification', fontsize=14, fontweight='bold')

metric_cols = ['Accuracy', 'F1', 'Precision', 'Recall', 'AUC-ROC']
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']

# Bar chart for all metrics
x = np.arange(len(metric_cols))
width = 0.18
for i, (_, row) in enumerate(metrics_df.iterrows()):
    vals = [row[m] for m in metric_cols]
    axes[0].bar(x + i*width, vals, width, label=row['Model'], color=colors[i], alpha=0.85)
axes[0].set_xticks(x + width * 1.5)
axes[0].set_xticklabels(metric_cols)
axes[0].set_ylim(0.5, 1.0)
axes[0].set_title('All Metrics Comparison')
axes[0].set_ylabel('Score')
axes[0].legend(fontsize=8, loc='lower right')
axes[0].grid(axis='y', alpha=0.3)

# AUC-ROC ranking
sorted_df = metrics_df.sort_values('AUC-ROC')
bars = axes[1].barh(sorted_df['Model'], sorted_df['AUC-ROC'],
                     color=colors[:len(sorted_df)][::-1], alpha=0.85)
axes[1].set_xlim(0.5, 1.0)
axes[1].set_title('AUC-ROC Ranking')
axes[1].set_xlabel('AUC-ROC Score')
for bar, val in zip(bars, sorted_df['AUC-ROC']):
    axes[1].text(val + 0.002, bar.get_y() + bar.get_height()/2,
                 f'{val:.4f}', va='center', fontsize=9)
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
save_fig('01_model_comparison')

  Saved → /content/drive/MyDrive/Dataset/results/ml_plots/01_model_comparison.png


## 16 — Confusion Matrices

In [19]:
def spark_conf_matrix(preds_df, model_name):
    cm_pd = preds_df.select('label', 'prediction') \
        .withColumn('label',      F.col('label').cast(IntegerType())) \
        .withColumn('prediction', F.col('prediction').cast(IntegerType())) \
        .toPandas()
    return confusion_matrix(cm_pd['label'], cm_pd['prediction'])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Confusion Matrices', fontsize=14, fontweight='bold')
class_labels = ['Negative', 'Positive']

for ax, (preds, title) in zip(axes, [
    (preds_lr_tab,  'LR — Tabular'),
    (preds_lr_comb, 'LR — Tabular+TF-IDF'),
    (preds_rf,      'Random Forest'),
]):
    cm = spark_conf_matrix(preds, title)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues',
                xticklabels=class_labels, yticklabels=class_labels, ax=ax)
    ax.set_title(title)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    # Add raw counts
    for i in range(2):
        for j in range(2):
            ax.text(j+0.5, i+0.7, f'({cm[i,j]:,})',
                    ha='center', va='center', fontsize=8, color='gray')

plt.tight_layout()
save_fig('02_confusion_matrices')

# XGBoost confusion matrix separately
fig, ax = plt.subplots(figsize=(5, 4))
cm_xgb = confusion_matrix(y_test_xgb, xgb_preds)
cm_xgb_norm = cm_xgb.astype('float') / cm_xgb.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_xgb_norm, annot=True, fmt='.2%', cmap='Oranges',
            xticklabels=class_labels, yticklabels=class_labels, ax=ax)
ax.set_title('XGBoost — Tabular (500k sample)')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout()
save_fig('03_confusion_xgboost')

  Saved → /content/drive/MyDrive/Dataset/results/ml_plots/02_confusion_matrices.png
  Saved → /content/drive/MyDrive/Dataset/results/ml_plots/03_confusion_xgboost.png


## 17 — ROC Curves

In [16]:
from pyspark.ml.functions import vector_to_array

def get_roc_from_spark(preds_df, name, sample_frac=0.1):
    roc_pd = preds_df.select(
        'label',
        vector_to_array('probability').getItem(1).alias('prob_pos')
    ).sample(fraction=sample_frac, seed=42).toPandas()
    fpr, tpr, _ = roc_curve(roc_pd['label'], roc_pd['prob_pos'])
    auc = roc_auc_score(roc_pd['label'], roc_pd['prob_pos'])
    return fpr, tpr, auc

plt.figure(figsize=(9, 7))

for preds, name, color in [
    (preds_lr_tab,  'LR — Tabular',          '#3498db'),
    (preds_lr_comb, 'LR — Tabular+TF-IDF',   '#2ecc71'),
    (preds_rf,      'Random Forest',           '#e74c3c'),
]:
    fpr, tpr, auc = get_roc_from_spark(preds, name)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color, linewidth=2)

# XGBoost ROC
fpr_xgb, tpr_xgb, _ = roc_curve(y_test_xgb, xgb_preds_proba)
auc_xgb = roc_auc_score(y_test_xgb, xgb_preds_proba)
plt.plot(fpr_xgb, tpr_xgb, label=f'XGBoost — sample (AUC={auc_xgb:.3f})',
         color='#f39c12', linewidth=2)

plt.plot([0,1], [0,1], 'k--', linewidth=1, label='Random Baseline')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — All Models', fontsize=14, fontweight='bold')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
save_fig('04_roc_curves')

  Saved → /content/drive/MyDrive/Dataset/results/ml_plots/04_roc_curves.png


## 18 — Random Forest Feature Importance

In [17]:
rf_stage = rf_model.stages[-1]  # RF model from pipeline
importances = rf_stage.featureImportances.toArray()

feat_imp = pd.DataFrame({
    'feature': NUMERIC_FEATURES,
    'importance': importances
}).sort_values('importance', ascending=False).head(20)

plt.figure(figsize=(11, 7))
plt.barh(feat_imp['feature'][::-1], feat_imp['importance'][::-1],
         color=sns.color_palette('viridis', 20))
plt.title('Random Forest — Top 20 Feature Importances', fontsize=14, fontweight='bold')
plt.xlabel('Feature Importance (Gini)')
plt.tight_layout()
save_fig('05_rf_feature_importance')

  Saved → /content/drive/MyDrive/Dataset/results/ml_plots/05_rf_feature_importance.png


## 19 — XGBoost Feature Importance

In [18]:
xgb_imp = pd.DataFrame({
    'feature': NUMERIC_FEATURES,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False).head(20)

plt.figure(figsize=(11, 7))
plt.barh(xgb_imp['feature'][::-1], xgb_imp['importance'][::-1],
         color=sns.color_palette('plasma', 20))
plt.title('XGBoost — Top 20 Feature Importances', fontsize=14, fontweight='bold')
plt.xlabel('Feature Importance (Gain)')
plt.tight_layout()
save_fig('06_xgb_feature_importance')

  Saved → /content/drive/MyDrive/Dataset/results/ml_plots/06_xgb_feature_importance.png


---
# Save Models & Results

## 20 — Save PySpark Models & Results

In [20]:
# Save PySpark models
lr_tab_model.write().overwrite().save(f'{MODELS_DIR}/lr_tabular_model')
print('Saved: lr_tabular_model')

lr_comb_model.write().overwrite().save(f'{MODELS_DIR}/lr_combined_model')
print('Saved: lr_combined_model')

rf_model.write().overwrite().save(f'{MODELS_DIR}/rf_tabular_model')
print('Saved: rf_tabular_model')

# Save XGBoost model
joblib.dump(xgb_model, f'{MODELS_DIR}/xgboost_model.pkl')
print('Saved: xgboost_model.pkl')

# Save metrics
metrics_path = f'{RESULTS_DIR}/ml_metrics.csv'
metrics_df.to_csv(metrics_path, index=False)
print(f'Saved: ml_metrics.csv')

# Save feature importance
feat_imp.to_csv(f'{RESULTS_DIR}/rf_feature_importance.csv', index=False)
xgb_imp.to_csv(f'{RESULTS_DIR}/xgb_feature_importance.csv', index=False)
print('Saved: feature importance CSVs')

print('\nAll models and results saved to Google Drive.')

Saved: lr_tabular_model
Saved: lr_combined_model
Saved: rf_tabular_model
Saved: xgboost_model.pkl
Saved: ml_metrics.csv
Saved: feature importance CSVs

All models and results saved to Google Drive.


---
## Results Summary

| Model | Features | Notes |
|-------|----------|-------|
| **Logistic Regression** | Tabular (34) | Baseline, full dataset, PySpark MLlib |
| **Logistic Regression** | Tabular + TF-IDF (34+10k) | Adds text signal, PySpark MLlib |
| **Random Forest** | Tabular (34) | 100 trees, depth 10, full dataset |
| **XGBoost** | Tabular (34) | 300 estimators, 500k sample |

### Next Notebooks
| Notebook | Content |
|----------|---------|
| `4_DL_LSTM_BERT.ipynb` | LSTM + BERT fine-tuning for sentiment |
| `5_TopicModeling.ipynb` | LDA + BERTopic on review text |
| `6_AnomalyDetection.ipynb` | Isolation Forest + Autoencoder (fake review detection) |
| `7_Recommendations.ipynb` | ALS collaborative filtering + content-based + K-Means |